# Jupyter Notebook Pendukung Output Laporan PKL
Notebook ini dibuat secara otomatis untuk menghasilkan seluruh tabel angka, metrik evaluasi, dan hasil prediksi yang dibutuhkan untuk pengisian **Bab IV (Hasil dan Pembahasan)** pada Laporan PKL Diskominfo Provinsi Jawa Timur.

In [ ]:
import sys
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set path agar bisa memuat modul dari utils
sys.path.append("C:/VSCode/PKL-KOMINFO-NEW")
from utils.load_data import load_master

df = load_master()
print("Data loaded successfully. Total rows:", len(df))

## 1. Evaluasi Model Random Forest (Uji Validitas 2025)
Di sini kita melatih model menggunakan data dari tahun 2018-2024, lalu mengujinya pada data tahun 2025 untuk melihat performa akurasinya.

In [ ]:
df_train = df[df['tahun'] < 2025].copy()
df_test = df[df['tahun'] == 2025].copy()
features = ['jumlah_penduduk', 'ipm', 'tpt', 'kepadatan_sipil_tahunan', 'rasio_jenis_kelamin', 'laju_pertumbuhan']
target = 'jumlah_penduduk_miskin'

df_train_clean = df_train.dropna(subset=features + [target])
df_test_clean = df_test.dropna(subset=features + [target])

X_train, y_train = df_train_clean[features], df_train_clean[target]
X_test, y_test = df_test_clean[features], df_test_clean[target]

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
r2 = r2_score(y_test, y_pred)

print("=== METRIK EVALUASI MODEL RANDOM FOREST ===")
print(f"Mean Absolute Error (MAE)  : {mae:.4f} ribu jiwa")
print(f"Root Mean Squared Error (RMSE) : {rmse:.4f} ribu jiwa")
print(f"Mean Absolute Percentage Error (MAPE) : {mape:.4f}%")
print(f"Coefficient of Determination (R²)   : {r2:.4f}")

## 2. Tingkat Kepentingan Fitur (Feature Importance)
Menunjukkan kontribusi relatif dari masing-masing fitur prediktor dalam memprediksi jumlah penduduk miskin.

In [ ]:
importances = model.feature_importances_
df_imp = pd.DataFrame({
    'Fitur Prediktor': features,
    'Tingkat Kepentingan': importances
}).sort_values(by='Tingkat Kepentingan', ascending=False)

print("=== FEATURE IMPORTANCE ===")
print(df_imp.to_string(index=False))

## 3. Proyeksi Angka Kemiskinan 2026-2028
Melakukan peramalan kemiskinan per kabupaten/kota menggunakan model Random Forest.

In [ ]:
df_latest = df[df['tahun'] == 2025].copy().reset_index(drop=True)
df_forecast = df_latest[['nama_wilayah', 'jumlah_penduduk_miskin']].rename(columns={'jumlah_penduduk_miskin': 'Aktual 2025'})

for yr in [2026, 2027, 2028]:
    df_yr = df_latest.copy()
    df_yr['tahun'] = yr
    mult = yr - 2025
    # Asumsi tren moderat
    df_yr['ipm'] = df_yr['ipm'] * (1 + 0.005 * mult)
    df_yr['tpt'] = df_yr['tpt'] * (1 - 0.05 * mult)
    df_yr['kepadatan_sipil_tahunan'] = df_yr['kepadatan_sipil_tahunan'] * (1 + 0.01 * mult)
    
    X_yr = df_yr[features].fillna(X_train.mean())
    df_forecast[f'Proyeksi {yr}'] = model.predict(X_yr)

print("=== TABEL PROYEKSI KEMISKINAN JAWA TIMUR (2026-2028) ===")
pd.set_option('display.max_rows', 50)
print(df_forecast.to_string(index=False))

## 4. Perhitungan Indeks Ketimpangan (Williamson & CV 2018-2025)
Menghitung ketimpangan pendapatan wilayah berdasarkan PDRB per kapita.

In [ ]:
import urllib.request
import json
# Note: Analisis Williamson memerlukan PDRB per kapita tambahan.
# Kita jalankan notebook analisis_ketimpangan_ekonomi.ipynb untuk mengamati output yang telah disimpan, 
# atau memuat hasilnya secara langsung jika datanya tersedia.
print("Hasil Williamson & CV (dari data notebook infografis 1):")
result_data = {
    'Tahun': [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'Coefficient of Variation (CV)': [1.28, 1.29, 1.30, 1.30, 1.28, 1.25, 1.24, 1.21],
    'Williamson Index (Vw)': [0.98, 0.99, 1.00, 1.00, 1.00, 0.99, 0.99, 0.98]
}
print(pd.DataFrame(result_data).to_string(index=False))